# Persona Steering Experiment

Run this notebook in Google Colab with a **GPU runtime** (A100 recommended, L4 may work).

**Runtime > Change runtime type > A100 GPU**

## How to run across multiple sessions

Each phase saves results to Google Drive. You can split across runtimes:

| Session | What to run | GPU? | Time |
|---------|------------|------|------|
| **1** | Setup + Phase 0 + Phase 1 generate + save | Yes | ~2 hrs |
| **2** (or local) | Phase 1 evaluate + plot, pick coefficients | No | ~10 min |
| **3** | Setup + Phase 2 generate + save | Yes | ~8 hrs |
| **4** (or local) | Phase 2 evaluate + geometry | No | ~20 min |
| **5** | Setup + Phase 3 generate + save | Yes | ~1 hr |
| **6** (or local) | Phase 3 evaluate | No | ~5 min |

**When resuming**: just run the Setup cell first (it restores results from Drive), then skip to wherever you left off.

## Setup

In [ ]:
# Clone repo and install
!git clone https://github.com/agastyasridharan/assistant-axis.git
%cd assistant-axis
!pip install -e . 2>&1 | tail -5

# Mount Google Drive for saving results across runtimes
from google.colab import drive
drive.mount("/content/drive")

# Restore previous results if resuming
import os
DRIVE_RESULTS = "/content/drive/MyDrive/persona_steering_results"
if os.path.exists(DRIVE_RESULTS):
    !cp -r {DRIVE_RESULTS}/* results/ 2>/dev/null || true
    print("Restored previous results from Drive")
else:
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    print("Fresh run — no previous results found")

In [ ]:
# Download pre-computed vectors from HuggingFace
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="lu-christina/assistant-axis-vectors",
    repo_type="dataset",
    local_dir="data/hf_vectors",
    allow_patterns=["gemma-2-27b/*"],
)
print("Done!")

In [ ]:
# Set your OpenAI API key (needed for evaluation only)
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # <-- paste your key here

In [ ]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Phase 0: Geometry Validation (no GPU needed)

Verifies that residual vectors (orthogonal to the Assistant Axis) carry persona-specific information.

In [ ]:
import json
import numpy as np
from pathlib import Path
from assistant_axis import load_axis
from assistant_axis.axis import compute_residual_vectors_batch

# Load
axis = load_axis("data/hf_vectors/gemma-2-27b/assistant_axis.pt")
vectors_dir = Path("data/hf_vectors/gemma-2-27b/role_vectors")
role_vectors = {}
for f in sorted(vectors_dir.glob("*.pt")):
    data = torch.load(f, map_location="cpu", weights_only=False)
    role_vectors[f.stem] = data if torch.is_tensor(data) else data["vector"]

print(f"Loaded {len(role_vectors)} role vectors, axis shape: {axis.shape}")

# Decompose at layer 22 (Gemma 2 27B target layer)
layer = 22
decomp = compute_residual_vectors_batch(role_vectors, axis, layer)

# Norm analysis
fracs = [d["residual_norm"] / d["full_norm"] for d in decomp.values()]
print(f"\nResidual fraction: {np.mean(fracs):.1%} +/- {np.std(fracs):.1%}")
print(f"(83% means the axis captures only 17% of role vector info)")

# Target roles
for role in ["ghost", "librarian", "demon", "angel", "assistant"]:
    if role in decomp:
        d = decomp[role]
        frac = d["residual_norm"] / d["full_norm"]
        print(f"  {role:12s}: axis_proj={d['proj_scalar']:+9.1f}  residual_frac={frac:.1%}")

## Phase 1: Pilot Coefficient Sweep

Tests 7 coefficient values x 3 steering methods x 3 personas on 50 questions. Takes ~2 hours on A100.

In [ ]:
!python scripts/persona_steering_experiment.py \
    --model google/gemma-2-27b-it \
    --axis data/hf_vectors/gemma-2-27b/assistant_axis.pt \
    --vectors_dir data/hf_vectors/gemma-2-27b/role_vectors \
    --default_vector data/hf_vectors/gemma-2-27b/default_vector.pt \
    --output_dir results/phase1_pilot \
    --phase pilot \
    --n_questions 50

In [ ]:
# Save Phase 1 generation results to Drive (in case runtime disconnects)
!cp -r results/phase1_pilot/ {DRIVE_RESULTS}/phase1_pilot/
print("Phase 1 generation results saved to Drive")

In [ ]:
# Score with LLM judge (needs OPENAI_API_KEY)
!python scripts/evaluate_steering.py \
    --results results/phase1_pilot/pilot_results.jsonl \
    --output results/phase1_pilot/scores.jsonl

In [ ]:
# Analyze dose-response curves
import jsonlines
import matplotlib.pyplot as plt
from collections import defaultdict

scored = []
with jsonlines.open("results/phase1_pilot/scores.jsonl") as reader:
    for r in reader:
        scored.append(r)

# Group by persona x method x coefficient
groups = defaultdict(list)
for r in scored:
    if r.get("role_score") is None or r.get("method") == "none":
        continue
    key = (r["persona"], r["method"], r["coefficient"])
    groups[key].append(r["role_score"])

# Plot dose-response
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
methods = ["role_vector", "residual", "random"]
colors = {"role_vector": "blue", "residual": "purple", "random": "gray",
          "norm_matched": "green", "mean_ablation": "orange"}

for ax, persona in zip(axes, ["ghost", "librarian", "demon"]):
    for method in ["role_vector", "residual", "random", "norm_matched"]:
        coeffs, means = [], []
        for (p, m, c), scores in sorted(groups.items()):
            if p == persona and m == method:
                coeffs.append(c)
                means.append(np.mean(scores))
        if coeffs:
            ax.plot(coeffs, means, "o-", label=method, color=colors.get(method, "black"))

    # Add baselines
    for r in scored:
        if r["persona"] == persona and r.get("condition") == "prompt_only" and r.get("role_score") is not None:
            baseline_scores = [s["role_score"] for s in scored
                             if s["persona"] == persona and s.get("condition") == "prompt_only"
                             and s.get("role_score") is not None]
            ax.axhline(np.mean(baseline_scores), color="red", linestyle="--", label="prompt-only baseline")
            break

    ax.set_title(persona.capitalize())
    ax.set_xlabel("Coefficient")
    ax.set_ylabel("Mean Judge Score (0-3)")
    ax.legend(fontsize=8)
    ax.set_ylim(-0.1, 3.1)
    ax.grid(True, alpha=0.3)

plt.suptitle("Phase 1: Dose-Response Curves", fontsize=14)
plt.tight_layout()
plt.savefig("results/phase1_pilot/dose_response.png", dpi=150)
plt.show()

# Print optimal coefficients
print("\nOptimal coefficients (highest role score with coherence >= 3.5):")
for persona in ["ghost", "librarian", "demon"]:
    best_coeff, best_score = 0, 0
    for (p, m, c), scores in groups.items():
        if p == persona and m == "role_vector":
            mean_s = np.mean(scores)
            # Check coherence
            coh = [r["coherence_score"] for r in scored
                   if r["persona"] == p and r["method"] == m and r["coefficient"] == c
                   and r.get("coherence_score") is not None]
            mean_coh = np.mean(coh) if coh else 5
            if mean_s > best_score and mean_coh >= 3.5:
                best_score = mean_s
                best_coeff = c
    print(f"  {persona}: coefficient={best_coeff}, score={best_score:.2f}")

## Phase 2: Main Experiment

**Edit the `optimal_coeffs` below** using the values from Phase 1 above.

In [ ]:
# EDIT THESE based on Phase 1 results
OPTIMAL_COEFFS = '{"ghost": 10, "librarian": 5, "demon": 15, "angel": 10}'

In [ ]:
!python scripts/persona_steering_experiment.py \
    --model google/gemma-2-27b-it \
    --axis data/hf_vectors/gemma-2-27b/assistant_axis.pt \
    --vectors_dir data/hf_vectors/gemma-2-27b/role_vectors \
    --default_vector data/hf_vectors/gemma-2-27b/default_vector.pt \
    --output_dir results/phase2_main \
    --phase main \
    --optimal_coeffs "$OPTIMAL_COEFFS"

In [ ]:
# Save Phase 2 generation results to Drive
!cp -r results/phase2_main/ {DRIVE_RESULTS}/phase2_main/
print("Phase 2 generation results saved to Drive")

In [ ]:
# Score with cross-persona evaluation
!python scripts/evaluate_steering.py \
    --results results/phase2_main/main_results.jsonl \
    --output results/phase2_main/scores.jsonl \
    --cross_persona

In [ ]:
# Geometric analysis
!python scripts/analyze_geometry.py \
    --vectors_dir data/hf_vectors/gemma-2-27b/role_vectors \
    --axis data/hf_vectors/gemma-2-27b/assistant_axis.pt \
    --results results/phase2_main/main_results.jsonl \
    --layer 22 \
    --output results/phase2_main/geometry.json \
    --plot results/phase2_main/geometry.html

## Phase 3: Multi-Turn Stability

In [ ]:
!python scripts/persona_steering_experiment.py \
    --model google/gemma-2-27b-it \
    --axis data/hf_vectors/gemma-2-27b/assistant_axis.pt \
    --vectors_dir data/hf_vectors/gemma-2-27b/role_vectors \
    --default_vector data/hf_vectors/gemma-2-27b/default_vector.pt \
    --output_dir results/phase3_multiturn \
    --phase multiturn \
    --optimal_coeffs "$OPTIMAL_COEFFS"

In [ ]:
# Save Phase 3 generation results to Drive
!cp -r results/phase3_multiturn/ {DRIVE_RESULTS}/phase3_multiturn/
print("Phase 3 generation results saved to Drive")

In [ ]:
!python scripts/evaluate_steering.py \
    --results results/phase3_multiturn/multiturn_results.jsonl \
    --output results/phase3_multiturn/scores.jsonl \
    --multiturn

## Download Results

Save everything to Google Drive or download directly.

In [ ]:
# Option 1: Save to Google Drive
from google.colab import drive
drive.mount("/content/drive")
!cp -r results/ /content/drive/MyDrive/persona_steering_results/

In [ ]:
# Option 2: Download as zip
!zip -r results.zip results/
from google.colab import files
files.download("results.zip")